# VR Chinese OCR Evaluation
### Deep Learning for Media — MPATE-GE 2039 / DM-GY 9103

**Goal:** Evaluate four OCR models on handwritten Chinese characters under nine VR-simulated perturbations to determine which model is best suited for a Meta Quest whiteboard application.

**Models evaluated:** ANCHOR · PaddleOCR · EasyOCR · CnOCR

**Dataset:** CASIA-HWDB (Kaggle) — offline handwritten Chinese characters

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so all modules are importable from the notebook
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter

print(f'Repo root: {repo_root}')

## 2. Load Data

Data is pre-generated and stored in `notebooks/data/`. Run `python regenerate_data.py` from the repo root if you need to rebuild these files.

**Test set composition:**
- ~818 clean samples (unperturbed baseline)
- ~88–107 samples per perturbation type (9 types)
- All labels are verified Chinese characters (CJK Unicode range)

In [ ]:
data_dir = Path('data')

X_train    = np.load(data_dir / 'X_train.npy',    allow_pickle=True)
X_test     = np.load(data_dir / 'X_test.npy',     allow_pickle=True)
y_train    = np.load(data_dir / 'y_train.npy',    allow_pickle=True)
y_test     = np.load(data_dir / 'y_test.npy',     allow_pickle=True)
pert_types = np.load(data_dir / 'pert_types.npy', allow_pickle=True)

print(f'Train: {len(X_train):,} samples')
print(f'Test:  {len(X_test):,} samples')
print(f'\nTest set breakdown:')
counts = Counter(str(p) for p in pert_types)
for k, v in sorted(counts.items()):
    label = 'clean' if k == 'None' else k
    print(f'  {label:30s}: {v} samples')

### Sample images

In [ ]:
# Show a few clean test samples with their ground truth labels
clean_mask = np.array([p is None for p in pert_types])
X_clean = X_test[clean_mask]
y_clean = y_test[clean_mask]

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(X_clean[i], cmap='gray')
    ax.set_title(y_clean[i], fontsize=14)
    ax.axis('off')
plt.suptitle('Sample clean test images with ground truth labels', y=1.02)
plt.tight_layout()
plt.show()

## 3. Perturbation Visualization

Each perturbation simulates a real condition encountered when using a Meta Quest headset to view a physical whiteboard.

In [ ]:
from perturbations.perturbation_analysis import show_perturbation_grid

show_perturbation_grid(
    images_raw=X_train,
    labels_raw=y_train,
    n_samples=5,
    save_path=None,
)

## 4. Model Inference

Each model is run on the full test set (clean + all 9 perturbation types). Predictions are cached to `../outputs/` so you only need to run inference once — subsequent notebook runs load from cache.

| Model | Architecture | Notes |
|---|---|---|
| **ANCHOR** | VGG-like CNN | Trained on CASIA-HWDB, 3755-class fixed vocabulary |
| **PaddleOCR** | Detection + recognition pipeline | Baidu production OCR, state of the art for Chinese |
| **EasyOCR** | CRAFT detector + CRNN | Open-source, Chinese support |
| **CnOCR** | DenseNet + GRU | Lightweight Chinese OCR library |

In [ ]:
from models.anchor_inference     import run_anchor
from models.paddleocr_inference  import run_paddleocr
from models.easyocr_inference    import run_easyocr
from models.cnocr_inference      import run_cnocr

outputs_dir = Path('../outputs')
outputs_dir.mkdir(exist_ok=True)

model_fns = {
    'ANCHOR':    run_anchor,
    'PaddleOCR': run_paddleocr,
    'EasyOCR':   run_easyocr,
    'CnOCR':     run_cnocr,
}

predictions = {}
for name, fn in model_fns.items():
    cache_path = outputs_dir / f'preds_{name.lower()}_full.npy'
    if cache_path.exists():
        predictions[name] = np.load(cache_path, allow_pickle=True)
        print(f'{name}: loaded from cache')
    else:
        print(f'{name}: running inference...')
        predictions[name] = fn(X_test)
        np.save(cache_path, predictions[name])
        print(f'{name}: done, saved to {cache_path}')

## 5. Evaluation

Metrics computed per model, overall and broken down by perturbation type:
- **Exact match accuracy** — top-1, predicted string equals ground truth
- **CER** — character error rate (lower is better)
- **Macro F1** — averaged across all character classes

In [ ]:
from evaluation.metrics import evaluate
from evaluation.utils   import print_results_table, print_perturbation_table, print_robustness_table

results = {
    name: evaluate(preds, y_test, pert_types)
    for name, preds in predictions.items()
}

print('=== OVERALL RESULTS ===')
print_results_table(results)

In [ ]:
print('=== ACCURACY BY PERTURBATION TYPE ===')
print_perturbation_table(results)

In [ ]:
print('=== ROBUSTNESS GAP (accuracy drop from clean baseline) ===')
print_robustness_table(results)

## 6. Figures

Visualizations for the paper. All figures are also saved to `../outputs/`.

In [ ]:
from evaluation.accuracy_curves import (
    plot_accuracy_by_perturbation,
    plot_robustness_gap_heatmap,
    plot_clean_comparison,
    plot_f1_by_perturbation,
)

plot_accuracy_by_perturbation(
    results,
    save_path=str(outputs_dir / 'fig_accuracy_by_perturbation.png')
)

In [ ]:
plot_robustness_gap_heatmap(
    results,
    save_path=str(outputs_dir / 'fig_robustness_heatmap.png')
)

In [ ]:
plot_clean_comparison(
    results,
    save_path=str(outputs_dir / 'fig_clean_comparison.png')
)

In [ ]:
plot_f1_by_perturbation(
    results,
    save_path=str(outputs_dir / 'fig_f1_by_perturbation.png')
)

## 7. Key Findings

**ANCHOR is the strongest model for VR handwriting recognition.** Despite being a 2013 model, it outperforms all three modern production OCR systems because it was purpose-built for isolated handwritten Chinese characters — the exact input type we have.

**VR-relevant perturbations (ranked by damage):**

| Perturbation | Most affected model | Least affected |
|---|---|---|
| Motion blur | PaddleOCR (−33pt) | CnOCR (−13pt) |
| Gaussian noise | ANCHOR (−32pt) | PaddleOCR (−0.5pt) |
| Low resolution | EasyOCR (−14pt) | PaddleOCR (−4pt) |
| Perspective warp | CnOCR (−4pt) | ANCHOR (improves) |

**Notable findings:**
- Motion blur is the single hardest perturbation — every model degrades significantly. This is the most important condition to address for a real VR deployment.
- Gaussian noise specifically destroys ANCHOR (−32 points) while barely affecting PaddleOCR (−0.5 points). ANCHOR relies on clean stroke patterns; noise corrupts them.
- ANCHOR is nearly unaffected by occlusion, suggesting its CNN learned to recognize partial strokes from the natural variation in real handwriting.
- TrOCR (no working Chinese handwriting checkpoint) and DINOv2 (<1% accuracy, insufficient per-class training data) were both ruled out — see the paper for details.

**Recommendation for the VR whiteboard app:** Use ANCHOR for highest accuracy on handwritten isolated characters. If noise robustness is a priority, PaddleOCR is the better choice — it is far more stable under image degradation.